In [ ]:
import sys
sys.path.append('..')

In [ ]:
import numpy as np
import pandas as pd
from os.path import join as pjoin
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from matplotlib import pyplot as plt
import copy
from utils.misc import load_config
import random

In [ ]:
root = '../data/multitask_processed'

In [ ]:
xPath = pjoin(root, 'X.h5')
yPath = pjoin(root, 'Y.h5')

In [ ]:
X = pd.read_hdf(xPath)
Y = pd.read_hdf(yPath)

In [ ]:
Y = pd.concat([Y, X.PHI], axis=1)
X = X.drop('PHI', axis=1)

In [ ]:
blind_well = random.sample(list(X.UWI.unique()), 10)

In [ ]:
orig_lat_long = [[X[X.UWI == i].lat.iloc[0], X[X.UWI == i].lng.iloc[0]] for i in X.UWI.unique()]
orig_lat_long = np.asarray(orig_lat_long)

In [ ]:
plt.scatter(orig_lat_long[:, 1], orig_lat_long[:, 0])

In [ ]:
Y = Y.LithID

In [ ]:
key, val = np.unique(Y, return_counts=True)

In [ ]:
data_config = load_config('..', 'config/data', 'base.yaml')

In [ ]:
lithology_classes = data_config['lithology_classes']
lithology_classes = {v: k for k, v in lithology_classes.items()}

In [ ]:
plt.bar([lithology_classes[k] for k in key], val)
plt.xticks(rotation=90)
plt.show()

In [ ]:
X.drop(['W_Tar', 'SW'], axis=1, inplace=True)

In [ ]:
lat_min, lat_max = X.lat.min(), X.lat.max()
lng_min, lng_max = X.lng.min(), X.lng.max()
depth_min, depth_max = X.DEPT.min(), X.DEPT.max()

In [ ]:
X.lat = (X.lat - lat_min) / (lat_max - lat_min)
X.lng = (X.lng - lng_min) / (lng_max - lng_min)
X.DEPT = (X.DEPT - depth_min) / (depth_max - depth_min)

In [ ]:
scaler = StandardScaler()
X.ILD = scaler.fit_transform(X.ILD.values.reshape(-1, 1))

In [ ]:
scaler_gr = StandardScaler()
X.GR = scaler_gr.fit_transform(X.GR.values.reshape(-1, 1))

scaler_nphi = StandardScaler()
X.NPHI = scaler_nphi.fit_transform(X.NPHI.values.reshape(-1, 1))

scaler_dphi = StandardScaler()
X.DPHI = scaler_dphi.fit_transform(X.DPHI.values.reshape(-1, 1))

scaler_phi = StandardScaler()
X.PHI = scaler_phi.fit_transform(X.PHI.values.reshape(-1, 1))

In [ ]:
blind_X = X[X.UWI.isin(blind_well) == True]
blind_Y = Y[X.UWI.isin(blind_well) == True]

Y = Y[X.UWI.isin(blind_well) == False]
X = X[X.UWI.isin(blind_well) == False]

In [ ]:
X_20 = copy.deepcopy(X)
Y_20 = copy.deepcopy(Y)

In [ ]:
# Haversine formula for distance calculation with rescaled coordinates
def haversine_scaled(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in kilometers
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c
    return distance

tgt_well = '00-07-20-079-04W4-0'
target_lat = 0.3597100651243003
target_lng = 0.8780005656429506

X_20['distance_to_target'] = X_20.apply(lambda row: haversine_scaled(row['lat'], row['lng'], target_lat, target_lng), axis=1)
X_20 = X_20[X_20['UWI'] != tgt_well].nsmallest(9224, 'distance_to_target')
Y_20 = Y_20[X_20.index]

temp_X = pd.read_hdf(xPath)
orig_lat_long_20 = [[temp_X[temp_X.UWI == i].lat.iloc[0], temp_X[temp_X.UWI == i].lng.iloc[0]] for i in X_20.UWI.unique()]
orig_lat_long_20 = np.asarray(orig_lat_long_20)

def calculate_polygon_area(coordinates):
    n = len(coordinates)
    area = 0.0
    for i in range(n):
        j = (i + 1) % n
        area += (coordinates[i][0] * coordinates[j][1]) - (coordinates[j][0] * coordinates[i][1])
    return 0.5 * abs(area)

plt.scatter(orig_lat_long_20[:, 1], orig_lat_long_20[:, 0])

In [ ]:
# from shapely.geometry import Polygon, MultiPoint
# multipoint = MultiPoint(orig_lat_long_20)
# convex_hull = multipoint.convex_hull
# conversion_factor = 111.32

# convex_hull.area * (conversion_factor ** 2)
# hull = ConvexHull(coordinates)

# area = hull.volume

# np.asarray(orig_lat_long_20).shape

In [ ]:
X_20 = X_20.drop(['UWI', 'distance_to_target'], axis=1)

In [ ]:
X = X.drop(['UWI'], axis=1)
blind_X = blind_X.drop(['UWI'], axis=1)

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(X, Y.LithID, test_size=0.2, random_state=42)
X_test, X_val, Y_test, Y_val = train_test_split(X_val, Y_val, test_size=0.5, random_state=42)

# X_train_20, X_test_20, Y_train_20, Y_test_20 = train_test_split(X_20, Y_20, test_size=0.15, random_state=42)

# 1. Random Forest - https://onepetro.org/SPEIOGS/proceedings/22AIS/1-22AIS/D011S003R001/515684

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=50, 
    random_state=42, 
    n_jobs = -1, 
    verbose = 1, 
    criterion = 'gini',
    bootstrap = False,
    max_depth = 20,
    min_samples_leaf = 2,
)

classifier.fit(X_train_20, Y_train_20)
Y_train_20_pred = classifier.predict(X_train_20)
Y_test_20_pred = classifier.predict(X_test_20)

In [ ]:
print('Accuracy Train: ', accuracy_score(Y_train_20, Y_train_20_pred))
print('Accuracy Test: ', accuracy_score(Y_test_20, Y_test_20_pred))
print('Accuracy Blind: ', accuracy_score(blind_Y, classifier.predict(blind_X)))

In [ ]:
from sklearn.neural_network import MLPClassifier

classifier = MLPClassifier(
    hidden_layer_sizes=(10, 10, 10),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size='auto',
    learning_rate='constant',
    learning_rate_init=0.001,
    power_t=0.5,
    max_iter=500,
    shuffle=True,
    random_state=42,
    tol=0.0001,
    verbose=True,
    warm_start=False,
    momentum=0.9
)

classifier.fit(X_train_20, Y_train_20)
Y_train_20_pred = classifier.predict(X_train_20)
Y_test_20_pred = classifier.predict(X_test_20)

In [ ]:
print('Accuracy Train: ', accuracy_score(Y_train_20, Y_train_20_pred))
print('Accuracy Test: ', accuracy_score(Y_test_20, Y_test_20_pred))
print('Accuracy Blind: ', accuracy_score(blind_Y, classifier.predict(blind_X)))

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=50, 
    random_state=42, 
    n_jobs = -1, 
    verbose = 1, 
    criterion = 'gini',
    bootstrap = False,
    max_depth = 20,
    min_samples_leaf = 2,
)

classifier.fit(X_train, Y_train)
Y_train_pred = classifier.predict(X_train)
Y_val_pred = classifier.predict(X_val)
Y_test_pred = classifier.predict(X_test)

In [ ]:
print('Accuracy Train: ', accuracy_score(Y_train, Y_train_pred))
print('Accuracy Val: ', accuracy_score(Y_val, Y_val_pred))
print('Accuracy Test: ', accuracy_score(Y_test, Y_test_pred))
print('Accuracy Blind: ', accuracy_score(blind_Y.LithID, classifier.predict(blind_X)))

In [ ]:
classifier = MLPClassifier(
    hidden_layer_sizes=(10, 10, 10),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size='auto',
    learning_rate='constant',
    learning_rate_init=0.001,
    power_t=0.5,
    max_iter=500,
    shuffle=True,
    random_state=42,
    tol=0.0001,
    verbose=True,
    warm_start=False,
    momentum=0.9
)

classifier.fit(X_train, Y_train)
Y_train_pred = classifier.predict(X_train)
Y_val_pred = classifier.predict(X_val)
Y_test_pred = classifier.predict(X_test)

In [ ]:
print('Accuracy Train: ', accuracy_score(Y_train, Y_train_pred))
print('Accuracy Val: ', accuracy_score(Y_val, Y_val_pred))
print('Accuracy Test: ', accuracy_score(Y_test, Y_test_pred))
print('Accuracy Blind: ', accuracy_score(blind_Y.LithID, classifier.predict(blind_X)))

# 1. XGB - https://onepetro.org/SPEIOGS/proceedings/22AIS/1-22AIS/D011S003R001/515684

In [ ]:
from xgboost import XGBClassifier

In [ ]:
classifier = XGBClassifier(
    n_estimators=90, 
    max_depth=6, 
    learning_rate=0.1, 
    random_state=42,
    colsample_bytree=1,
    Subsample=1,
)

classifier.fit(X_train_20, Y_train_20)
Y_train_20_pred = classifier.predict(X_train_20)
Y_test_20_pred = classifier.predict(X_test_20)

print('Accuracy Train: ', accuracy_score(Y_train_20, Y_train_20_pred))
print('Accuracy Test: ', accuracy_score(Y_test_20, Y_test_20_pred))
print('Accuracy Blind: ', accuracy_score(blind_Y, classifier.predict(blind_X)))

In [ ]:
classifier = XGBClassifier(
    n_estimators=90, 
    max_depth=6, 
    learning_rate=0.1, 
    random_state=42,
    colsample_bytree=1,
    Subsample=1,
)

classifier.fit(X_train, Y_train)
Y_train_pred = classifier.predict(X_train)
Y_val_pred = classifier.predict(X_val)
Y_test_pred = classifier.predict(X_test)

print('Accuracy Train: ', accuracy_score(Y_train, Y_train_pred))
print('Accuracy Val: ', accuracy_score(Y_val, Y_val_pred))
print('Accuracy Test: ', accuracy_score(Y_test, Y_test_pred))
print('Accuracy Blind: ', accuracy_score(blind_Y, classifier.predict(blind_X)))